In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics import classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset

In [ ]:
MODEL_NAME = "jhu-clsp/mmBERT-base" if os.getenv("KAGGLE_KERNEL_RUN_TYPE") else "/kaggle/input/jhu-clsp/mmBERT-base"
DATA = "/kaggle/input/competitions/rmit-hackathon-2026"

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I = {l: i for i, l in enumerate(LABELS)}
I2L = {i: l for i, l in enumerate(LABELS)}

MAXLEN = 256
EPOCHS = 3
LR = 2e-5
BS_PER_GPU = 16

# 2. Load Dataset
train = pd.read_csv(f"{DATA}/train.csv")

# 80% Train / 20% Validation Split
tr, va = train_test_split(
    train, 
    test_size=0.20, 
    stratify=train["label"], 
    random_state=42
)

# 3. Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_hf_dataset(df):
    d = df[["text"]].copy()
    d["labels"] = df["label"].map(L2I).values
    return (
        Dataset.from_pandas(d, preserve_index=False)
        .map(
            lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAXLEN),
            batched=True,
            remove_columns=["text"]
        )
    )

ds_tr = build_hf_dataset(tr)
ds_va = build_hf_dataset(va)

print(f"Dataset summary | Train (80%): {len(ds_tr)} | Val (20%): {len(ds_va)}")

# 4. Load Model (Configured for 3-class classification)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=I2L,
    label2id=L2I,
    attn_implementation="sdpa"  # Built-in PyTorch Scaled Dot-Product Attention
)

# 5. Compute Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_micro = f1_score(labels, preds, average="micro")
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro
    }

# 6. Training Arguments
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS_PER_GPU,
    per_device_eval_batch_size=BS_PER_GPU * 2,
    learning_rate=LR,
    weight_decay=0.01,
    eval_strategy="epoch",  # Fixed argument name
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none"
)

# 7. Trainer Setup & Execution
# 7. Trainer Setup & Execution
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tr,
    eval_dataset=ds_va,
    processing_class=tokenizer,  # Updated from tokenizer=tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Start fine-tuning
trainer.train()